# 第 28 课｜行情频率与场景

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`28_youtube-Lh-qXaEe4Ok/index.html`  
> **所属模块**：P1-M6 · 数据契约与策略报告

## 本节量化问题

Tick、分钟和日线如何聚合，信号时点如何避免看到未来？

## 准确定义与策略报告位置

OHLCV 重采样分别使用首、最高、最低、末和求和；标签代表区间起点还是终点必须明确。

- **核心关系**：OHLCV 聚合：Open=first，High=max，Low=min，Close=last，Volume=sum。
- **写入报告**：频率、时区、聚合和信号可得时点
- **P2 衔接**：P2 第 35–39 课与 L2：重采样、缺失和时点对齐。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 把逐笔样例聚合为 1 分钟 OHLCV

`label='right', closed='right'` 表示时间标签是区间右端；这直接影响信号何时可得。

In [2]:
ticks = pd.read_csv(DATA_DIR / "intraday_ticks.csv", parse_dates=["timestamp"]).set_index("timestamp")
one_minute = ticks.resample("1min", label="right", closed="right").agg(
    open=("price", "first"),
    high=("price", "max"),
    low=("price", "min"),
    close=("price", "last"),
    volume=("quantity", "sum"),
).dropna()

display(ticks.head(10))
display(one_minute.head())
assert (one_minute["high"] >= one_minute[["open", "close"]].max(axis=1)).all()
assert (one_minute["low"] <= one_minute[["open", "close"]].min(axis=1)).all()

,price,quantity,symbol
timestamp,,,
2025-01-02 09:30:01+08:00,99.99,76,SIM_A
2025-01-02 09:30:06+08:00,99.99,285,SIM_A
2025-01-02 09:30:11+08:00,99.99,201,SIM_A
2025-01-02 09:30:16+08:00,100.00,287,SIM_A
2025-01-02 09:30:21+08:00,100.01,279,SIM_A
2025-01-02 09:30:26+08:00,100.01,247,SIM_A
2025-01-02 09:30:31+08:00,100.00,208,SIM_A
2025-01-02 09:30:36+08:00,100.00,70,SIM_A
2025-01-02 09:30:41+08:00,100.01,129,SIM_A


,open,high,low,close,volume
timestamp,,,,,
2025-01-02 09:31:00+08:00,99.99,100.01,99.99,100.01,2369
2025-01-02 09:32:00+08:00,100.00,100.03,99.99,100.03,1529
2025-01-02 09:33:00+08:00,100.02,100.02,99.99,100.00,1797
2025-01-02 09:34:00+08:00,99.99,100.00,99.97,99.97,1422
2025-01-02 09:35:00+08:00,99.96,99.99,99.95,99.97,1934


## 方法边界与阶段验收

未闭合 K 线不可用于声称已知的收盘信号；跨频率合并要使用当时可得数据。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
